---
title: "Section 3: Where are nightlife patrons coming and going?"
format:
  html:
    code-fold: true
    code-summary: "Show code"
---

This section will extract Origin-Destination MRT flows via an API call from Land Transport Authority (LTA) Datamall to explore where nightlife patrons leave to. This will inform key recommendations given to LTA about potential night service transit routes for these nightlife patrons.

- Unfortunately the LTA Origin-Destination flows are split into two separate datasets for MRT stations and Bus stops, despite having a unified fare system where travellers generally combine both modes of transit in one trip.
- Based on a 2015 report by [UITP Asia-Pacific Centre for Transport Excellence](https://www.uitp.org/wp-content/uploads/sites/7/2025/04/The-Governance-of-PT-Market-Singapore-report.pdf), multi-modal trips combining both MRT and bus make up 24.8% of public transport modal share. MRT-only trips comprise 12%, while bus-only trips comprise 16.5%.
- **As a proxy, only MRT O-D flows will be used for analysis** because there are less nodes and therefore it is more generalizable for policy recommendations. Furthermore, MRT is usually used for longer-distance commutes which  might better capture patron's trips back home from nightlife clusters, rather than to other areas of Singapore.

# 1. Set up environment

In [1]:
import io
import os
from getpass import getpass
import warnings
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import plotly.express as px
from pathlib import Path
import altair as alt
import networkx as nx
from IPython.display import display

warnings.filterwarnings("ignore")

print("GeoPandas version:", gpd.__version__)

GeoPandas version: 1.1.3


# 2. Data wrangling of MRT nodes in nightlife clusters
This is to extract the MRT nodes within/nearest to nightlife clusters. This makes up the `nightlife_mrt` object.

In [3]:
venues_svy21 = gpd.read_file("../../data/nightlife_venues_walk.geojson").to_crs(epsg=3414)
clusters_wgs84 = gpd.read_file("../../data/nightlife_clusters.geojson").to_crs(epsg=4326)
clusters_svy21 = clusters_wgs84.to_crs(epsg=3414)
mrt_wgs84 = gpd.read_file("../../data/mrt_stations_clean.geojson").to_crs(epsg=4326)
mrt_svy21 = mrt_wgs84.to_crs(epsg=3414)

First, the MRT point data is spatially joined (inner join) to each cluster so that only MRT stations within the clusters are kept, and each MRT station has the associated cluster_id in a column.

In [12]:
nightlife_mrt = gpd.sjoin(
    mrt_svy21,
    clusters_svy21[["cluster_id", "geometry"]],
    how="inner",
    predicate="within",
).drop(columns=["index_right"])

nightlife_mrt.head(10)

,station_code,line_name,typ_cd,geometry,cluster_id
80,TE18,TEC,34,POINT (29179.97 29227.155),0
82,DT18,DTL,34,POINT (29668.873 29413.662),0
83,EW14 / NS26,EWNS,34,POINT (30018.537 29608.376),0
84,DT19 / NE4,DTL / NEL,34,POINT (29181.089 29652.704),0
85,NE5,NEL,34,POINT (29477.042 30102.117),0
88,EW13 / NS25,EWNS,34,POINT (30146.65 30591.49),1
89,CC3,CCL,34,POINT (30481.707 30627.452),1
90,CC2,CCL,34,POINT (29929.14 31022.162),1
91,DT21,DTL,34,POINT (29882.826 31230.844),1
92,CC1 / NE6 / NS24,CCL / NEL / NSL,34,POINT (29379.77 31303.854),1


In [52]:
# reproject
clusters_map_gdf = clusters_wgs84.to_crs(epsg=4326).copy()
mrt_map_gdf = nightlife_mrt.to_crs(epsg=4326).copy()

# initialize base map centered on singapore with Esri Dark Gray Base
m_all_stations = folium.Map(
    location=[1.3521, 103.8198],
    zoom_start=11.5,
    tiles=None,
)

folium.TileLayer(
    tiles="https://services.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin, © OpenStreetMap contributors, and the GIS User Community",
    name="Esri Dark Gray Base",
).add_to(m_all_stations)

# cluster polygon
cluster_fg = folium.FeatureGroup(name="Nightlife Clusters (MRT Coverage)")

for _, row in clusters_map_gdf.iterrows():
    fill_color = "#1f78b4"
    
    popup_html = f"""
    <b>Cluster ID:</b> {row['cluster_id']}<br>
    """
    
    folium.GeoJson(
        row["geometry"],
        style_function=lambda _, col=fill_color: {
            "fillColor": col,
            "color": col,
            "weight": 2,
            "fillOpacity": 0.30,
        },
        tooltip=f"Cluster {row['cluster_id']}",
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(cluster_fg)

cluster_fg.add_to(m_all_stations)

# add MRT stations inside clusters point layer
mrt_fg = folium.FeatureGroup(name="MRT Stations in Clusters")

for _, row in mrt_map_gdf.iterrows():
    pt = row.geometry.centroid  # handles both Point and Polygon geometries safely
    
    # include station name if available alongside station_code
    stn_name = row.get("station_name", row.get("STN_NAME", ""))
    label = f"{row['station_code']} ({stn_name})" if stn_name else f"{row['station_code']}"
    
    folium.CircleMarker(
        location=[pt.y, pt.x],
        radius=6,
        color="#ffffff",
        weight=1.5,
        fill=True,
        fill_color="#ff7f00",
        fill_opacity=0.95,
        tooltip=f"{label} — Cluster {row['cluster_id']}",
        popup=folium.Popup(
            f"<b>Station:</b> {label}<br><b>Cluster ID:</b> {row['cluster_id']}",
            max_width=250,
        ),
    ).add_to(mrt_fg)

mrt_fg.add_to(m_all_stations)

# add floating map title with data sources
title_html = """
<div style="position: fixed; top: 15px; left: 55px; z-index: 9999;
            background: rgba(25, 25, 25, 0.9); color: #f0f0f0; padding: 10px 16px;
            border: 1px solid #555; border-radius: 6px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.4);">
  <div style="font-size: 15px; font-weight: 700; letter-spacing: 0.2px;">
    MRT stations within Top 7 Nightlife Clusters in Singapore
  </div>
  <div style="font-size: 11px; color: #bbbbbb; margin-top: 2px;">
    Sources: LTA DataMall & OpenStreetMap (OSM)
  </div>
</div>
"""
m_all_stations.get_root().html.add_child(folium.Element(title_html))

# add legend
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999; background: rgba(25, 25, 25, 0.9);
            color: #f0f0f0; padding: 10px 14px; border: 1px solid #555; border-radius: 6px; font-size: 12px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.4);">
  <b>Nightlife Clusters & MRT Nodes</b><br>
  <span style="display:inline-block;width:12px;height:12px;background:#1f78b4;opacity:0.5;margin-right:6px;"></span>Cluster with MRT (O-D Node)<br>
  <span style="display:inline-block;width:12px;height:12px;background:#e31a1c;opacity:0.5;margin-right:6px;"></span>Cluster without MRT (Desert)<br>
  <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#ff7f00;margin-right:7px;"></span>MRT Station in Cluster
</div>
"""
m_all_stations.get_root().html.add_child(folium.Element(legend_html))

m_all_stations

**Troubleshooting**

So there's a few issues: because of the cluster shapes/the far distance of clusters from MRTs, only Clusters 0, 1 and 6 have MRT stations within the cluster. Mapping our O-D flows based on this existing list of MRTs is not sufficient to capture patterns for all nightlife districts we identified.

Therefore for these remaining clusters, we will need to use the OSM network to extract at least 1 MRT station per cluster, that is nearest to the cluster.

In [24]:
import networkx as nx
import osmnx as ox

In [25]:
# load the saved OSM pedestrian network and project to SVY21 (EPSG:3414)
G_walk = ox.load_graphml("../../data/nightlife_walk.graphml")
G_svy21 = ox.project_graph(G_walk, to_crs="EPSG:3414")

# tag existing in-cluster MRT matches
in_cluster_mrt = nightlife_mrt.copy()
in_cluster_mrt["assignment_type"] = "Within Cluster"
in_cluster_mrt["network_dist_m"] = 0.0

# identify clusters with 0 internal MRT stations
covered_clusters = set(in_cluster_mrt["cluster_id"].unique())
missing_clusters = clusters_svy21[~clusters_svy21["cluster_id"].isin(covered_clusters)].copy()

# snap all MRT stations to the nearest pedestrian network node
mrt_pts = mrt_svy21.copy()
mrt_pts["centroid"] = mrt_pts.geometry.centroid
mrt_pts["osm_node"] = ox.nearest_nodes(
    G_svy21,
    X=mrt_pts["centroid"].x.values,
    Y=mrt_pts["centroid"].y.values,
)

# find shortest walking path from each missing cluster centroid to MRT stations
nearest_records = []

for _, cl_row in missing_clusters.iterrows():
    cl_id = cl_row["cluster_id"]
    cl_cent = cl_row.geometry.centroid
    
    # snap cluster centroid to pedestrian network
    origin_node = ox.nearest_nodes(G_svy21, X=cl_cent.x, Y=cl_cent.y)
    
    # compute shortest network walking distances (in meters) from cluster to all reachable nodes
    lengths = nx.single_source_dijkstra_path_length(G_svy21, origin_node, weight="length")
    
    # map network distance to every MRT station
    mrt_candidate = mrt_pts.copy()
    mrt_candidate["network_dist_m"] = mrt_candidate["osm_node"].map(lengths)
    mrt_candidate = mrt_candidate.dropna(subset=["network_dist_m"])
    
    # select the closest station (keeps all line codes if the nearest station is an interchange)
    min_dist = mrt_candidate["network_dist_m"].min()
    closest_stns = mrt_candidate[mrt_candidate["network_dist_m"] == min_dist].copy()
    
    closest_stns["cluster_id"] = cl_id
    closest_stns["assignment_type"] = "Nearest Network"
    nearest_records.append(closest_stns.drop(columns=["centroid", "osm_node"]))

# combine in-cluster and nearest-network MRTs into updated nightlife_mrt
nearest_mrt_gdf = gpd.GeoDataFrame(
    pd.concat(nearest_records, ignore_index=True),
    crs=mrt_svy21.crs,
)

nightlife_mrt_expanded = gpd.GeoDataFrame(
    pd.concat([in_cluster_mrt, nearest_mrt_gdf], ignore_index=True),
    crs=mrt_svy21.crs,
)

# summary table per cluster
nightlife_mrt_expanded[
    ["cluster_id", "station_code", "assignment_type", "network_dist_m"]
].sort_values(["cluster_id", "network_dist_m"])

,cluster_id,station_code,assignment_type,network_dist_m
0,0,TE18,Within Cluster,0.000000
1,0,DT18,Within Cluster,0.000000
2,0,EW14 / NS26,Within Cluster,0.000000
3,0,DT19 / NE4,Within Cluster,0.000000
4,0,NE5,Within Cluster,0.000000
5,1,EW13 / NS25,Within Cluster,0.000000
6,1,CC3,Within Cluster,0.000000
7,1,CC2,Within Cluster,0.000000
8,1,DT21,Within Cluster,0.000000
9,1,CC1 / NE6 / NS24,Within Cluster,0.000000


In [51]:
# reproject layers to WGS84 for Folium
clusters_map_gdf = clusters_wgs84.to_crs(epsg=4326).copy()
mrt_expanded_wgs84 = nightlife_mrt_expanded.to_crs(epsg=4326).copy()

# initialize base map with Esri Dark Gray Base
m_expanded_stations = folium.Map(
    location=[1.3521, 103.8198],
    zoom_start=11.5,
    tiles=None,
)

folium.TileLayer(
    tiles="https://services.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin, © OpenStreetMap contributors, and the GIS User Community",
    name="Esri Dark Gray Base",
).add_to(m_expanded_stations)

# add cluster polygons (blue = has internal MRT, red = assigned nearest network MRT)
for _, row in clusters_map_gdf.iterrows():
    cl_id = row["cluster_id"]
    has_internal = cl_id in covered_clusters
    fill_color = "#1f78b4" if has_internal else "#e31a1c"
    status_label = "Internal MRT" if has_internal else "Nearest Network MRT Assigned"

    cl_stns = mrt_expanded_wgs84[mrt_expanded_wgs84["cluster_id"] == cl_id]
    stn_list = ", ".join(sorted(cl_stns["station_code"].unique()))

    popup_html = f"""
    <b>Cluster ID:</b> {cl_id}<br>
    <b>Coverage Type:</b> {status_label}<br>
    <b>Assigned MRT(s):</b> {stn_list}
    """

    folium.GeoJson(
        row["geometry"],
        style_function=lambda _, col=fill_color: {
            "fillColor": col,
            "color": col,
            "weight": 2,
            "fillOpacity": 0.30,
        },
        tooltip=f"Cluster {cl_id} ({status_label})",
        popup=folium.Popup(popup_html, max_width=300),
    ).add_to(m_expanded_stations)

# add MRT station markers (orange = within cluster, cyan = nearest network)
for _, row in mrt_expanded_wgs84.iterrows():
    pt = row.geometry.centroid
    is_internal = row["assignment_type"] == "Within Cluster"
    marker_color = "#ff7f00" if is_internal else "#00e5ff"

    stn_name = row.get("station_name", row.get("STN_NAME", ""))
    label = f"{row['station_code']} ({stn_name})" if stn_name else f"{row['station_code']}"
    dist_text = "Within cluster" if is_internal else f"{row['network_dist_m']:.0f} m walk"

    folium.CircleMarker(
        location=[pt.y, pt.x],
        radius=6,
        color="#ffffff",
        weight=1.5,
        fill=True,
        fill_color=marker_color,
        fill_opacity=0.95,
        tooltip=f"{label} — Cluster {row['cluster_id']} ({dist_text})",
        popup=folium.Popup(
            f"<b>Station:</b> {label}<br>"
            f"<b>Cluster ID:</b> {row['cluster_id']}<br>"
            f"<b>Assignment:</b> {row['assignment_type']} ({dist_text})",
            max_width=260,
        ),
    ).add_to(m_expanded_stations)

# add floating map title with data sources
title_html = """
<div style="position: fixed; top: 15px; left: 55px; z-index: 9999;
            background: rgba(25, 25, 25, 0.9); color: #f0f0f0; padding: 10px 16px;
            border: 1px solid #555; border-radius: 6px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.4);">
  <div style="font-size: 15px; font-weight: 700; letter-spacing: 0.2px;">
    MRT stations serving Top 7 Nightlife clusters in Singapore
  </div>
  <div style="font-size: 11px; color: #bbbbbb; margin-top: 2px;">
    Sources: LTA DataMall & OpenStreetMap (OSM)
  </div>
</div>
"""
m_expanded_stations.get_root().html.add_child(folium.Element(title_html))

# add clean legend
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999; background: rgba(25, 25, 25, 0.9);
            color: #f0f0f0; padding: 10px 14px; border: 1px solid #555; border-radius: 6px; font-size: 12px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.4);">
  <b>Nightlife Clusters & Assigned MRT Nodes</b><br>
  <span style="display:inline-block;width:12px;height:12px;background:#1f78b4;opacity:0.6;margin-right:6px;"></span>Cluster with Internal MRT<br>
  <span style="display:inline-block;width:12px;height:12px;background:#e31a1c;opacity:0.6;margin-right:6px;"></span>Cluster without Internal MRT<br>
  <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#ff7f00;margin-right:7px;"></span>MRT Within Cluster<br>
  <span style="display:inline-block;width:10px;height:10px;border-radius:50%;background:#00e5ff;margin-right:7px;"></span>Nearest MRT (OSM Network)
</div>
"""
m_expanded_stations.get_root().html.add_child(folium.Element(legend_html))

m_expanded_stations

Great! Now all nightlife clusters have at least 1 MRT station within/near it. There are a total of 17 stations in this list. This object `nightlife_mrt_expanded` is used as our final list of MRTs serving nightlife clusters in Singapore.

# 3. Data gathering and cleaning of Origin-Destination flows
Next, we will be mapping late-night O-D flows originating from MRT stations serving nightlife clusters, during nightlife operating hours. We will be able to find out where people go after nightlife activities.

The dataset returns number of trips by weekdays and weekends from the origin to destination bus stops. Sample: 2018-05, WEEKDAY, 17, TRAIN, CC28, CC1-NE6-NS24, 111 

From the API call, we extract O-D data particularly for

1. The whole month of October 2026
2. Times from 8pm (when bars start to close, and nightclubs start to open) to 12am (when transit stops)
3. ORIGIN_PT_CODE == "`nightlife_mrt_expanded`"


Ideally I would like to include more months but for computational efficiency we will keep to October 2026.

---

**Pulling the API from [LTA Datamall Dynamic Dataset folder](https://datamall.lta.gov.sg/content/datamall/en/dynamic-data.html)**

Following the instructions in the User Guide, I first downloaded Postman and pasted in the URL and my API Key for this dataset. It gave me the relevant code I needed to put in Python. Then, I filtered according to the 3 conditions I made above.

It is important to note that my station code names (especially for the tricky interchanges) in `nightlife_mrt_expanded` match that of the O-D dataset that I pull via an API. From the [API User Guide](https://datamall.lta.gov.sg/content/dam/datamall/datasets/LTA_DataMall_API_User_Guide.pdf), we can see the syntax lists examples for ORIGIN_PT_CODE as CC1-NE6-NS24. However, in my station_code column of `nightlife_mrt_expanded`, it could be formatted as CC1 / NE6 / NS24, or even in a different order like NS24 / CC1 / NE6. To fix this, we will extract the first station code (every letter before the first " " space), and make sure it is **within** ORIGIN_PT_CODE in the dataset.

In [33]:
import zipfile

In [56]:
api_key = os.environ.get("LTA_API_KEY", "").strip()
if not api_key:
    api_key = getpass("LTA DataMall API key (hidden): ").strip()

url = "https://datamall2.mytransport.sg/ltaodataservice/PV/ODTrain"
headers = {
    "AccountKey": api_key
}
params = {
    "Date": "202610"
}

response = requests.get(url, headers=headers, params=params)
response.raise_for_status()

download_link = response.json()["value"][0]["Link"]

# download and extract the zipped CSV in memory
zip_response = requests.get(download_link)
zip_response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(zip_response.content)) as zf:
    csv_filename = zf.namelist()[0]
    with zf.open(csv_filename) as f:
        od_train_raw = pd.read_csv(f)

# standardize column names and whitespace
od_train_raw.columns = od_train_raw.columns.str.strip()
od_train_raw["ORIGIN_PT_CODE"] = od_train_raw["ORIGIN_PT_CODE"].astype(str).str.strip()
od_train_raw["DESTINATION_PT_CODE"] = od_train_raw["DESTINATION_PT_CODE"].astype(str).str.strip()

# extract the first station code (everything before the first space) from nightlife_mrt_expanded
nightlife_mrt_expanded["first_code"] = (
    nightlife_mrt_expanded["station_code"]
    .astype(str)
    .str.strip()
    .str.split(" ")
    .str[0]
)

# build a lookup dictionary mapping each station's first_code to its cluster_id
code_to_cluster = dict(
    zip(nightlife_mrt_expanded["first_code"], nightlife_mrt_expanded["cluster_id"])
)
target_codes = set(code_to_cluster.keys())

# match first_code against dash-separated tokens in ORIGIN_PT_CODE (e.g., "CC1" in ["CC1", "NE6", "NS24"])
def match_cluster_id(od_code_str):
    tokens = od_code_str.split("-")
    for token in tokens:
        if token in code_to_cluster:
            return code_to_cluster[token]
    return None

# filter for October 2026 and nightlife origin stations
od_oct_origins = od_train_raw[
    od_train_raw["YEAR_MONTH"] == "2026-10"
].copy()

od_oct_origins["origin_cluster_id"] = od_oct_origins["ORIGIN_PT_CODE"].apply(match_cluster_id)
od_oct_origins = od_oct_origins.dropna(subset=["origin_cluster_id"]).copy()
od_oct_origins["origin_cluster_id"] = od_oct_origins["origin_cluster_id"].astype(int)

# filter for late-night window from 8pm to 12am (20:00 to 00:00)
late_night_hours = [20, 21, 22, 23, 0]
od_nightlife_late = od_oct_origins[
    od_oct_origins["TIME_PER_HOUR"].isin(late_night_hours)
].copy()

print(f"Total October 2026 rows for nightlife origin stations (all hours): {len(od_oct_origins):,}")
print(f"Filtered 8pm–12am rows for nightlife origin stations: {len(od_nightlife_late):,}")
od_nightlife_late.head(10)

LTA DataMall API key (hidden):  ········


Total October 2026 rows for nightlife origin stations (all hours): 55,934
Filtered 8pm–12am rows for nightlife origin stations: 13,405


,YEAR_MONTH,DAY_TYPE,TIME_PER_HOUR,PT_TYPE,ORIGIN_PT_CODE,DESTINATION_PT_CODE,TOTAL_TRIPS,origin_cluster_id
148,2026-10,WEEKDAY,20,TRAIN,DT18,TE25,3,0
244,2026-10,WEEKENDS/HOLIDAY,21,TRAIN,NE5,NS20,54,0
368,2026-10,WEEKDAY,21,TRAIN,CC3,DT13,94,1
455,2026-10,WEEKDAY,22,TRAIN,NE5,TE15,17,0
547,2026-10,WEEKENDS/HOLIDAY,22,TRAIN,NE5,TE27,21,0
750,2026-10,WEEKDAY,23,TRAIN,NE5,EW7,13,0
766,2026-10,WEEKENDS/HOLIDAY,20,TRAIN,EW9,NS17/CC15,44,4
794,2026-10,WEEKDAY,23,TRAIN,DT18,DT28,124,0
826,2026-10,WEEKENDS/HOLIDAY,22,TRAIN,NE5,SW5,68,0
895,2026-10,WEEKENDS/HOLIDAY,20,TRAIN,DT13,SE3,4,1


In [44]:
print(f"Total number of nighttime trips made in Oct 2026: {od_nightlife_late['TOTAL_TRIPS'].sum():,}")

Total number of nighttime trips made in Oct 2026: 759,721


In [35]:
# save O-D dataset
od_nightlife_late.to_csv("../../data/od_nightlife_late.csv", index=False)

In [36]:
import branca.colormap as cm

In [48]:
# extract first station code from mrt_wgs84 to match dash-separated LTA O-D tokens
mrt_plot = mrt_wgs84.to_crs(epsg=4326).copy()
mrt_plot["first_code"] = (
    mrt_plot["station_code"]
    .astype(str)
    .str.strip()
    .str.split(" ")
    .str[0]
)

# aggregate total late-night trips by destination and origin station codes
dest_grouped = (
    od_nightlife_late.groupby("DESTINATION_PT_CODE", as_index=False)["TOTAL_TRIPS"]
    .sum()
)
orig_grouped = (
    od_nightlife_late.groupby("ORIGIN_PT_CODE", as_index=False)["TOTAL_TRIPS"]
    .sum()
)

# build token-level lookup dictionaries so interchange ordering differences still match
dest_token_trips = {}
for _, row in dest_grouped.iterrows():
    for token in str(row["DESTINATION_PT_CODE"]).split("-"):
        dest_token_trips[token] = dest_token_trips.get(token, 0) + row["TOTAL_TRIPS"]

orig_token_trips = {}
for _, row in orig_grouped.iterrows():
    for token in str(row["ORIGIN_PT_CODE"]).split("-"):
        orig_token_trips[token] = orig_token_trips.get(token, 0) + row["TOTAL_TRIPS"]

# map aggregated trip counts onto mrt_plot
mrt_plot["dest_trips"] = mrt_plot["first_code"].map(dest_token_trips).fillna(0)
mrt_plot["orig_trips"] = mrt_plot["first_code"].map(orig_token_trips).fillna(0)

# deduplicate physical interchange points if mrt_wgs84 stores one row per line code
stn_name_col = next(
    (c for c in ["station_name", "STN_NAME", "NAME", "name"] if c in mrt_plot.columns),
    None,
)
if stn_name_col:
    mrt_plot = (
        mrt_plot.sort_values(["dest_trips", "orig_trips"], ascending=False)
        .drop_duplicates(subset=[stn_name_col])
        .copy()
    )

# flipped color scales: light-to-deep blue for destinations, light-to-deep red for origins
dest_max = mrt_plot["dest_trips"].max()
orig_max = mrt_plot["orig_trips"].max()

dest_cmap = cm.LinearColormap(
    colors=["#caf0f8", "#48cae4", "#0096c7", "#1d4e89"],
    vmin=0,
    vmax=dest_max,
)
orig_cmap = cm.LinearColormap(
    colors=["#fff3b0", "#faa307", "#e85d04", "#9b2226"],
    vmin=0,
    vmax=orig_max,
)

# initialize Esri Dark Gray Base map centered on Singapore
m_od_mrt = folium.Map(
    location=[1.3521, 103.8198],
    zoom_start=11.5,
    tiles=None,
)

folium.TileLayer(
    tiles="https://services.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Dark_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, HERE, Garmin, © OpenStreetMap contributors, and the GIS User Community",
    name="Esri Dark Gray Base",
    control=False,
).add_to(m_od_mrt)

# layer for destination MRT stations colored and sized by total incoming nightlife trips
dest_fg = folium.FeatureGroup(name="Destination MRTs (Alighting Trips)", show=True)

for _, row in mrt_plot.iterrows():
    pt = row.geometry.centroid
    trips = row["dest_trips"]
    label = f"{row['station_code']} ({row[stn_name_col]})" if stn_name_col else row["station_code"]

    color = dest_cmap(trips) if trips > 0 else "#4a4a4a"
    radius = 4 + (10 * (trips / dest_max) ** 0.5) if trips > 0 else 3.5

    folium.CircleMarker(
        location=[pt.y, pt.x],
        radius=radius,
        color="#d9f0ff" if trips > 0 else "#666666",
        weight=1 if trips > 0 else 0.5,
        fill=True,
        fill_color=color,
        fill_opacity=0.9 if trips > 0 else 0.4,
        tooltip=f"Destination: {label} | {int(trips):,} trips",
        popup=folium.Popup(
            f"<b>Destination MRT:</b> {label}<br>"
            f"<b>Late-Night Alighting Trips (8pm–12am):</b> {int(trips):,}",
            max_width=280,
        ),
    ).add_to(dest_fg)

dest_fg.add_to(m_od_mrt)

# layer for nightlife origin MRT stations colored and sized by total outgoing trips
orig_fg = folium.FeatureGroup(name="Nightlife Origin MRTs (Boarding Trips)", show=True)

origin_only_gdf = mrt_plot[mrt_plot["orig_trips"] > 0].sort_values("orig_trips", ascending=True)

for _, row in origin_only_gdf.iterrows():
    pt = row.geometry.centroid
    trips = row["orig_trips"]
    label = f"{row['station_code']} ({row[stn_name_col]})" if stn_name_col else row["station_code"]

    color = orig_cmap(trips)
    radius = 7 + (11 * (trips / orig_max) ** 0.5)

    folium.CircleMarker(
        location=[pt.y, pt.x],
        radius=radius,
        color="#ffffff",
        weight=2,
        fill=True,
        fill_color=color,
        fill_opacity=0.95,
        tooltip=f"Nightlife Origin: {label} | {int(trips):,} trips",
        popup=folium.Popup(
            f"<b>Nightlife Origin MRT:</b> {label}<br>"
            f"<b>Late-Night Boarding Trips (8pm–12am):</b> {int(trips):,}",
            max_width=280,
        ),
    ).add_to(orig_fg)

orig_fg.add_to(m_od_mrt)

# add layer control so origin and destination layers can be toggled
folium.LayerControl(position="bottomright", collapsed=False).add_to(m_od_mrt)

# add floating map title
title_html = """
<div style="position: fixed; top: 15px; left: 55px; z-index: 9999;
            background: rgba(20, 20, 20, 0.92); color: #f0f0f0; padding: 10px 16px;
            border: 1px solid #555; border-radius: 6px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.5);">
  <div style="font-size: 15px; font-weight: 700; letter-spacing: 0.2px;">
    Top Origins and Destinations from Top 7 Nightlife Clusters in Singapore
  </div>
  <div style="font-size: 11px; color: #bbbbbb; margin-top: 2px;">
    LTA DataMall MRT Passenger Volume (October 2026, 8:00 PM – 12:00 AM)
  </div>
</div>
"""
m_od_mrt.get_root().html.add_child(folium.Element(title_html))

# add collapsible legend box with toggle button
legend_html = f"""
<div id="od-legend-box" style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: rgba(20, 20, 20, 0.92); color: #f0f0f0; padding: 10px 14px;
            border: 1px solid #555; border-radius: 6px; font-size: 12px;
            box-shadow: 0 2px 6px rgba(0,0,0,0.5); max-width: 280px;">
  <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px;">
    <b>Late-Night MRT O–D Volume (8pm–12am)</b>
    <button onclick="var body = document.getElementById('od-legend-body');
                     if (body.style.display === 'none') {{ body.style.display = 'block'; this.innerText = 'Hide'; }}
                     else {{ body.style.display = 'none'; this.innerText = 'Show'; }}"
            style="background: #333; color: #fff; border: 1px solid #666; border-radius: 4px;
                   padding: 2px 6px; font-size: 11px; cursor: pointer; margin-left: 8px;">
      Hide
    </button>
  </div>
  <div id="od-legend-body">
    <div style="margin-top: 6px;"><b>Destination MRTs (Alighting Trips)</b></div>
    <div style="height: 10px; width: 100%; border-radius: 3px; margin: 4px 0;
                background: linear-gradient(to right, #caf0f8, #48cae4, #0096c7, #1d4e89);"></div>
    <div style="display: flex; justify-content: space-between; font-size: 11px; color: #ccc;">
      <span>1</span>
      <span>{int(dest_max):,} trips</span>
    </div>
    <div style="font-size: 11px; color: #999; margin-top: 2px;">
      <span style="display:inline-block;width:8px;height:8px;border-radius:50%;background:#4a4a4a;margin-right:5px;"></span>0 trips
    </div>

    <div style="margin-top: 10px;"><b>Nightlife Origin MRTs (Boarding Trips)</b></div>
    <div style="height: 10px; width: 100%; border-radius: 3px; margin: 4px 0;
                background: linear-gradient(to right, #fff3b0, #faa307, #e85d04, #9b2226);"></div>
    <div style="display: flex; justify-content: space-between; font-size: 11px; color: #ccc;">
      <span>Low</span>
      <span>{int(orig_max):,} trips</span>
    </div>
    <div style="font-size: 11px; color: #aaa; margin-top: 4px;">
      *Circle area is proportional to total trips; white-ringed circles indicate nightlife origin stations.
    </div>
  </div>
</div>
"""
m_od_mrt.get_root().html.add_child(folium.Element(legend_html))

m_od_mrt

The map highlights quite interesting patterns. 

1. **Origins**: The most number of trips emerge from NE5 (Clarke Quay), with 122,873 out of 759,721 trips made in Oct 2026. This verifies how Clarke Quay is responsible for most of (17%) of Singapore's nightlife, and is part of the largest Cluster 0. DT23 (Bendemeer) is responsible for the least number of trips. Looking at the previous map above, it quite poorly serves Cluster 6 because it sits along the edge of the cluster; patrons might opt for other closer MRT stations, or rely on bus/ride-hail alternatives

What's to be noted is that government authorities trialed [extended liquor sales](https://www.channelnewsasia.com/singapore/boat-clarke-quay-alcohol-liquor-hours-4am-6323531) in Clarke Quay till 4am, which was formalized August of this year. This has definitely increased traffic in the area late at night which also could be better supported with extended transit hours from this destination.

2. **Destinations**: Most destinations tend to be along the Downtown line East-bound, such as DT33 (Tampines East) and DT28 (Kaki Bukit). This is reasonable  because Tampines is the most populous neighborhood in Singapore. What's interesting is that Tampines and Kaki Bukit (in the wider area of Bedok) are both mature estates, which means they were established a lot earlier and probably have older residents. It would be interesting to look at the demographics of nightlife patrons.

### _Limitations:_

This O-D trip data combines MRT and LRT stations into the same dataset, but LRT stations should be weighted less because they are local rail systems connecting MRT stations to the neighborhoods. This will understate total passenger demand at the LRT's parent MRT hubs. For example, NE15 (Buangkok) is a destination responsible for 11,371 trips when in reality it also serves the neighborhoods covered by the many Sengkang and Punggol LRT.
In the future, I should instead add up the LRT trips and add them into the nearest MRT station instead, as a proxy for the approximate end destination.

---
In Section 4, I will translate all my past findings into actionable insights and recommendations for the Singapore government.